# 멀티모달 조합 임베딩 테스트

`types.Content`에 여러 모달리티를 함께 넣어 임베딩할 수 있는지 테스트합니다.

| 방식 | 임베딩 입력 | 목적 |
|---|---|---|
| D. PDF + 텍스트 | 1페이지 PDF + `get_text()` | PDF 레이아웃 + 텍스트 보강 |
| E. 이미지 + 텍스트 | 페이지 PNG + `get_text()` | 시각 정보 + 텍스트 보강 |
| F. PDF + 이미지 | 1페이지 PDF + 페이지 PNG | 두 가지 시각 표현 조합 |

기존 비교(`test_embedding_comparison.ipynb`)의 단일 모달리티 결과와 비교합니다:
- A. PDF 바이트 (Top-1 평균: 0.4366)
- B. 텍스트만 (Top-1 평균: 0.6441)
- C. 페이지 이미지 (Top-1 평균: 0.4499)

In [1]:
import os
import time
import io
import numpy as np
import faiss
import fitz
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

EMBEDDING_MODEL = "gemini-embedding-2-preview"
EMBEDDING_DIM = 3072
PDF_PATH = "data/헤리티지 역사와 과학 제58권 제4호(통권 제110권).pdf"

test_queries = [
    "어떤 연구들을 진행했었는가?",
    "탕건에 대해서 과학적 분석 결과가 어떻게 되었는가?",
    "두 자락치마의 조선 왕족 실록의 기록은?",
    "훈장 가운데에 있는 명칭은?",
]

print("설정 완료")

설정 완료


## 페이지 데이터 준비 (PDF 바이트, 텍스트, 이미지)

In [2]:
doc = fitz.open(PDF_PATH)
total_pages = len(doc)

# PDF 바이트 (1페이지짜리 PDF)
page_pdfs = []
for i in range(total_pages):
    single = fitz.open()
    single.insert_pdf(doc, from_page=i, to_page=i)
    page_pdfs.append(single.tobytes())
    single.close()

# 텍스트
page_texts = []
for i in range(total_pages):
    text = doc[i].get_text().strip()
    page_texts.append(text if text else " ")

# 페이지 이미지 (PNG)
page_images = []
for i in range(total_pages):
    mat = fitz.Matrix(150 / 72, 150 / 72)
    pix = doc[i].get_pixmap(matrix=mat)
    page_images.append(pix.tobytes("png"))

doc.close()
print(f"총 {total_pages}페이지 준비 완료")
print(f"  PDF 바이트: 첫 페이지 {len(page_pdfs[0])/1024:.0f}KB")
print(f"  텍스트: 첫 페이지 {len(page_texts[0])}자")
print(f"  이미지: 첫 페이지 {len(page_images[0])/1024:.0f}KB")

총 260페이지 준비 완료
  PDF 바이트: 첫 페이지 93KB
  텍스트: 첫 페이지 178자
  이미지: 첫 페이지 331KB


## 공통 함수

In [3]:
def embed_batch(contents_list: list, batch_size: int = 5, label: str = "") -> np.ndarray:
    """contents 리스트를 배치로 임베딩"""
    all_embeddings = []
    total = len(contents_list)
    t0 = time.time()

    for i in range(0, total, batch_size):
        batch = contents_list[i : i + batch_size]
        try:
            response = client.models.embed_content(
                model=EMBEDDING_MODEL,
                contents=batch,
                config=types.EmbedContentConfig(
                    output_dimensionality=EMBEDDING_DIM,
                    task_type="RETRIEVAL_DOCUMENT",
                ),
            )
            for emb in response.embeddings:
                all_embeddings.append(emb.values)
        except Exception as e:
            print(f"  [{label}] 배치 {i}~{i+batch_size} 실패: {e}")
            # 실패한 배치는 개별로 재시도
            for j, item in enumerate(batch):
                try:
                    resp = client.models.embed_content(
                        model=EMBEDDING_MODEL,
                        contents=[item],
                        config=types.EmbedContentConfig(
                            output_dimensionality=EMBEDDING_DIM,
                            task_type="RETRIEVAL_DOCUMENT",
                        ),
                    )
                    all_embeddings.append(resp.embeddings[0].values)
                except Exception as e2:
                    print(f"    페이지 {i+j+1} 개별 실패: {e2}")
                    all_embeddings.append([0.0] * EMBEDDING_DIM)  # 제로 벡터 폴백
        if (i // batch_size) % 10 == 0:
            print(f"  [{label}] {min(i+batch_size, total)}/{total}")

    elapsed = time.time() - t0
    arr = np.array(all_embeddings, dtype=np.float32)
    print(f"  [{label}] 완료: {arr.shape} ({elapsed:.1f}초)")
    return arr

def build_index(embeddings: np.ndarray) -> faiss.IndexFlatIP:
    """FAISS 인덱스 구축"""
    emb = embeddings.copy()
    faiss.normalize_L2(emb)
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    index.add(emb)
    return index

def embed_query(query: str) -> np.ndarray:
    """쿼리 텍스트 임베딩"""
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=query,
        config=types.EmbedContentConfig(
            output_dimensionality=EMBEDDING_DIM,
            task_type="RETRIEVAL_QUERY",
        ),
    )
    emb = np.array([response.embeddings[0].values], dtype=np.float32)
    faiss.normalize_L2(emb)
    return emb

def search_index(index: faiss.IndexFlatIP, query_emb: np.ndarray, top_k: int = 5):
    """인덱스에서 검색"""
    scores, indices = index.search(query_emb, top_k)
    return [(int(idx) + 1, float(score)) for idx, score in zip(indices[0], scores[0])]

print("공통 함수 정의 완료")

공통 함수 정의 완료


## 테스트 1: 단일 페이지로 API 호환성 확인

260페이지 전체를 임베딩하기 전에, 먼저 1페이지로 3가지 조합이 API에서 동작하는지 확인합니다.

In [4]:
# 1페이지로 3가지 멀티모달 조합 테스트
test_page = 95  # 0-indexed (96페이지 - 텍스트+이미지 있는 페이지)

combinations = {
    "D. PDF+텍스트": types.Content(parts=[
        types.Part.from_bytes(data=page_pdfs[test_page], mime_type="application/pdf"),
        types.Part.from_text(text=page_texts[test_page]),
    ]),
    "E. 이미지+텍스트": types.Content(parts=[
        types.Part.from_bytes(data=page_images[test_page], mime_type="image/png"),
        types.Part.from_text(text=page_texts[test_page]),
    ]),
    "F. PDF+이미지": types.Content(parts=[
        types.Part.from_bytes(data=page_pdfs[test_page], mime_type="application/pdf"),
        types.Part.from_bytes(data=page_images[test_page], mime_type="image/png"),
    ]),
}

print(f"테스트 페이지: {test_page + 1}")
print(f"  텍스트 길이: {len(page_texts[test_page])}자")
print(f"  PDF 크기: {len(page_pdfs[test_page])/1024:.0f}KB")
print(f"  이미지 크기: {len(page_images[test_page])/1024:.0f}KB")
print()

for label, content in combinations.items():
    try:
        t0 = time.time()
        response = client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=content,
            config=types.EmbedContentConfig(
                output_dimensionality=EMBEDDING_DIM,
                task_type="RETRIEVAL_DOCUMENT",
            ),
        )
        elapsed = time.time() - t0
        emb = response.embeddings[0].values
        print(f"  {label}: OK (차원={len(emb)}, {elapsed:.2f}초)")
    except Exception as e:
        print(f"  {label}: FAIL - {e}")

테스트 페이지: 96
  텍스트 길이: 1120자
  PDF 크기: 442KB
  이미지 크기: 183KB

  D. PDF+텍스트: OK (차원=3072, 1.72초)
  E. 이미지+텍스트: OK (차원=3072, 1.22초)
  F. PDF+이미지: OK (차원=3072, 1.45초)


## 전체 페이지 임베딩 (성공한 조합만 실행)

위 테스트에서 성공한 조합에 대해 260페이지 전체를 임베딩합니다.

In [5]:
# D. PDF + 텍스트
print("=== D. PDF + 텍스트 임베딩 ===")
contents_d = [
    types.Content(parts=[
        types.Part.from_bytes(data=page_pdfs[i], mime_type="application/pdf"),
        types.Part.from_text(text=page_texts[i]),
    ])
    for i in range(total_pages)
]
emb_d = embed_batch(contents_d, label="PDF+텍스트")
index_d = build_index(emb_d)

=== D. PDF + 텍스트 임베딩 ===
  [PDF+텍스트] 5/260
  [PDF+텍스트] 55/260
  [PDF+텍스트] 105/260
  [PDF+텍스트] 155/260
  [PDF+텍스트] 205/260
  [PDF+텍스트] 255/260
  [PDF+텍스트] 완료: (260, 3072) (127.2초)


In [6]:
# E. 이미지 + 텍스트
print("=== E. 이미지 + 텍스트 임베딩 ===")
contents_e = [
    types.Content(parts=[
        types.Part.from_bytes(data=page_images[i], mime_type="image/png"),
        types.Part.from_text(text=page_texts[i]),
    ])
    for i in range(total_pages)
]
emb_e = embed_batch(contents_e, label="이미지+텍스트")
index_e = build_index(emb_e)

=== E. 이미지 + 텍스트 임베딩 ===
  [이미지+텍스트] 5/260
  [이미지+텍스트] 55/260
  [이미지+텍스트] 105/260
  [이미지+텍스트] 155/260
  [이미지+텍스트] 205/260
  [이미지+텍스트] 255/260
  [이미지+텍스트] 완료: (260, 3072) (85.2초)


In [7]:
# F. PDF + 이미지
print("=== F. PDF + 이미지 임베딩 ===")
contents_f = [
    types.Content(parts=[
        types.Part.from_bytes(data=page_pdfs[i], mime_type="application/pdf"),
        types.Part.from_bytes(data=page_images[i], mime_type="image/png"),
    ])
    for i in range(total_pages)
]
emb_f = embed_batch(contents_f, label="PDF+이미지")
index_f = build_index(emb_f)

=== F. PDF + 이미지 임베딩 ===
  [PDF+이미지] 5/260
  [PDF+이미지] 55/260
  [PDF+이미지] 105/260
  [PDF+이미지] 155/260
  [PDF+이미지] 205/260
  [PDF+이미지] 255/260
  [PDF+이미지] 완료: (260, 3072) (127.0초)


## 비교 검색 실행

In [8]:
# 기존 단일 모달리티 기준값 (test_embedding_comparison 결과)
baseline = {
    "A. PDF만": 0.4366,
    "B. 텍스트만": 0.6441,
    "C. 이미지만": 0.4499,
}

# 멀티모달 조합 인덱스 (성공한 것만)
multimodal_indices = {}
if 'index_d' in dir(): multimodal_indices["D. PDF+텍스트"] = index_d
if 'index_e' in dir(): multimodal_indices["E. 이미지+텍스트"] = index_e
if 'index_f' in dir(): multimodal_indices["F. PDF+이미지"] = index_f

for i, query in enumerate(test_queries, 1):
    print(f"\n{'='*80}")
    print(f"쿼리 {i}: {query}")
    print(f"{'='*80}")

    query_emb = embed_query(query)

    print(f"\n{'방식':<16} | {'#1':>20} | {'#2':>20} | {'#3':>20} | {'#4':>20} | {'#5':>20}")
    print(f"{'':-<16}-+-{'':->20}-+-{'':->20}-+-{'':->20}-+-{'':->20}-+-{'':->20}")

    for label, idx in multimodal_indices.items():
        results = search_index(idx, query_emb)
        cols = [f"p{page}({score:.4f})" for page, score in results]
        print(f"{label:<16} | {cols[0]:>20} | {cols[1]:>20} | {cols[2]:>20} | {cols[3]:>20} | {cols[4]:>20}")


쿼리 1: 어떤 연구들을 진행했었는가?

방식               |                   #1 |                   #2 |                   #3 |                   #4 |                   #5
-----------------+----------------------+----------------------+----------------------+----------------------+---------------------
D. PDF+텍스트       |         p226(0.4612) |         p144(0.4548) |          p58(0.4547) |          p72(0.4540) |         p106(0.4448)
E. 이미지+텍스트       |          p72(0.4727) |          p58(0.4675) |         p226(0.4567) |         p144(0.4552) |         p106(0.4472)
F. PDF+이미지       |         p129(0.4059) |         p226(0.4017) |         p130(0.4015) |           p5(0.3993) |          p43(0.3990)

쿼리 2: 탕건에 대해서 과학적 분석 결과가 어떻게 되었는가?

방식               |                   #1 |                   #2 |                   #3 |                   #4 |                   #5
-----------------+----------------------+----------------------+----------------------+----------------------+---------------------
D. PDF+텍스트     

## 종합 요약: 단일 vs 멀티모달 조합 비교

In [ ]:
print("="*60)
print("전체 쿼리 평균 Top-1 / Top-3 / Top-5 유사도")
print("="*60)

# 기존 단일 모달리티 기준값
print("\n[기존 단일 모달리티 - test_embedding_comparison 결과]")
for label, score in baseline.items():
    print(f"  {label:<16} Top-1 평균: {score:.4f}")

# 멀티모달 조합
print("\n[멀티모달 조합 - 이번 테스트 결과]")
for label, idx in multimodal_indices.items():
    top1_scores = []
    top3_scores = []
    top5_scores = []
    for query in test_queries:
        query_emb = embed_query(query)
        results = search_index(idx, query_emb)
        top1_scores.append(results[0][1])
        top3_scores.append(np.mean([s for _, s in results[:3]]))
        top5_scores.append(np.mean([s for _, s in results[:5]]))

    print(f"\n  {label}")
    print(f"    Top-1 평균: {np.mean(top1_scores):.4f}")
    print(f"    Top-3 평균: {np.mean(top3_scores):.4f}")
    print(f"    Top-5 평균: {np.mean(top5_scores):.4f}")

print("\n" + "="*60)
print("참고: B. 텍스트만 = 0.6441 이 현재까지 최고 성적")
print("멀티모달 조합이 이를 넘는지가 핵심 포인트")
print("="*60)